# 5. On-Policy Monte Carlo Control ($\epsilon$-Greedy)
### Improves the Policy Being Followed

**On-Policy Monte Carlo Control** evaluates and improves the **exact same policy** that is used to make decisions. To prevent premature convergence to suboptimal deterministic actions, it employs an **$\epsilon$-soft policy** (such as $\epsilon$-greedy).

#### Mathematical Formulation:
1. **$\epsilon$-Greedy Policy**:
   $$\pi(a \mid s) = \begin{cases} 1 - \epsilon + \frac{\epsilon}{|\mathcal{A}|} & \text{if } a = \arg\max_{a'} Q(s, a') \\ \frac{\epsilon}{|\mathcal{A}|} & \text{if } a \neq \arg\max_{a'} Q(s, a') \end{cases}$$
2. **Policy Improvement Guarantee**:
   Any $\epsilon$-greedy policy with respect to $q_\pi$ is guaranteed to be an improvement over any $\epsilon$-soft policy $\pi$:
   $$q_\pi(s, \pi'(s)) = \sum_a \pi'(a \mid s) q_\pi(s, a) \ge V^\pi(s)$$


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


In [ ]:
# Step 2: Define a Pure Python Blackjack Environment (Sutton & Barto Ch. 5)
# State: (player_sum: 12..21, dealer_showing_card: 1..10, usable_ace: bool)
# Actions: 0 = Stick, 1 = Hit

class BlackjackEnv:
    def __init__(self):
        self.actions = [0, 1]  # 0: Stick, 1: Hit
        
    def _draw_card(self):
        card = np.random.randint(1, 14)
        return min(card, 10)  # Face cards (J, Q, K) have value 10; Ace is 1
        
    def _draw_hand(self):
        return [self._draw_card(), self._draw_card()]
        
    def _get_hand_value(self, hand):
        val = sum(hand)
        has_ace = 1 in hand
        if has_ace and val + 10 <= 21:
            return val + 10, True  # Usable Ace
        return val, False
        
    def reset(self, start_state=None, start_action=None):
        if start_state is not None:
            player_sum, dealer_card, usable_ace = start_state
            self.player_hand = [1, player_sum - 11] if usable_ace else [player_sum // 2, player_sum - player_sum // 2]
            self.dealer_hand = [dealer_card, self._draw_card()]
        else:
            self.player_hand = self._draw_hand()
            while self._get_hand_value(self.player_hand)[0] < 12:
                self.player_hand.append(self._draw_card())
            self.dealer_hand = self._draw_hand()
            
        p_val, usable_ace = self._get_hand_value(self.player_hand)
        return (p_val, self.dealer_hand[0], usable_ace)
        
    def step(self, action):
        if action == 1:  # Hit
            self.player_hand.append(self._draw_card())
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            if p_val > 21:
                return (p_val, self.dealer_hand[0], usable_ace), -1.0, True  # Bust
            return (p_val, self.dealer_hand[0], usable_ace), 0.0, False
        else:  # Stick: Dealer plays
            while True:
                d_val, _ = self._get_hand_value(self.dealer_hand)
                if d_val >= 17:
                    break
                self.dealer_hand.append(self._draw_card())
                
            p_val, usable_ace = self._get_hand_value(self.player_hand)
            d_val, _ = self._get_hand_value(self.dealer_hand)
            
            if d_val > 21 or p_val > d_val:
                reward = 1.0  # Player wins
            elif p_val == d_val:
                reward = 0.0  # Draw
            else:
                reward = -1.0 # Dealer wins
                
            return (p_val, self.dealer_hand[0], usable_ace), reward, True

env = BlackjackEnv()
print("Blackjack environment initialized.")


In [ ]:
# Step 3: Implement On-Policy ε-Greedy MC Control
def on_policy_mc_control(env, num_episodes=80000, epsilon_start=1.0, epsilon_min=0.05, decay_rate=0.99995):
    Q = defaultdict(lambda: np.zeros(len(env.actions)))
    N = defaultdict(lambda: np.zeros(len(env.actions)))
    
    eps = epsilon_start
    episode_rewards = []
    
    for ep in range(1, num_episodes + 1):
        eps = max(epsilon_min, eps * decay_rate)
        
        # Behavior policy: epsilon-greedy
        episode = []
        s = env.reset()
        while True:
            if np.random.rand() < eps:
                a = np.random.choice(env.actions)
            else:
                a = np.argmax(Q[s])
                
            next_s, r, done = env.step(a)
            episode.append((s, a, r))
            s = next_s
            if done:
                episode_rewards.append(r)
                break
                
        # First-visit update of Q
        G = 0.0
        for t in reversed(range(len(episode))):
            s_t, a_t, r_t = episode[t]
            G = 1.0 * G + r_t
            if (s_t, a_t) not in [(step[0], step[1]) for step in episode[:t]]:
                N[s_t][a_t] += 1
                Q[s_t][a_t] += (G - Q[s_t][a_t]) / N[s_t][a_t]
                
    return Q, episode_rewards

Q_on, rewards = on_policy_mc_control(env, num_episodes=70000)


In [ ]:
# Step 4: Plot Moving Average of Rewards during Training
window = 2000
smooth_rewards = np.convolve(rewards, np.ones(window)/window, mode='valid')

plt.figure(figsize=(9, 4))
plt.plot(smooth_rewards, color='darkgreen')
plt.xlabel("Episode")
plt.ylabel(f"Average Reward (Window={window})")
plt.title("On-Policy ε-Greedy MC Control: Training Reward Curve")
plt.grid(True)
plt.show()


### Summary
- Continues exploration continuously through the $\epsilon$-soft guarantee.
- Finds the optimal $\epsilon$-soft policy rather than the true unconstrained optimal policy.
